In [1]:
import torch
torch.cuda.is_available(), torch.__version__

(True, '2.9.1+cu128')

In [2]:
x = torch.arange(4, dtype=float)
x

tensor([0., 1., 2., 3.], dtype=torch.float64)

In [3]:
x.requires_grad = True

In [4]:
y = 2 * torch.dot(x, x)
y

tensor(28., dtype=torch.float64, grad_fn=<MulBackward0>)

In [5]:
y.backward()

In [6]:
x.grad

tensor([ 0.,  4.,  8., 12.], dtype=torch.float64)

In [7]:
x.grad == 4 * x

tensor([True, True, True, True])

In [8]:
import numpy as np
num_states = 7
i_to_n = {}
i_to_n["0"] = "C1"
i_to_n["1"] = "C2"
i_to_n["2"] = "C3"
i_to_n["3"] = "Pass"
i_to_n["4"] = "Pub"
i_to_n["5"] = "FB"
i_to_n["6"] = "Sleep"
n_to_i = {}
for i, name in i_to_n.items():
    n_to_i[name] = int(i)

In [9]:
Pss = [
    # C1 C2 C3 Pass Pub FB Sleep
    [.0, .5, .0, .0, .0, .5, .0], # C1
    [.0, .0, .8, .0, .0, .0, .2], # C2
    [.0, .0, .0, .6, .4, .0, .0], # C3
    [.0, .0, .0, .0, .0, .0, 1.], # Pass
    [.2, .4, .4, .0, .0, .0, .0], # Pub
    [.1, .0, .0, .0, .0, .9, .0], # FB
    [.0, .0, .0, .0, .0, .0, 1.], # Sleep
]
Pss = np.array(Pss)
rewards = [-2, -2, -2, 10, 1, -1, 0]
gama = .5

In [10]:
def compute_return(start_index = 0, chain = None, gamma=.5) -> float:
    retrn, power, gamma = .0, 0, gamma
    for i in range(start_index, len(chain)):
        retrn += np.power(gamma, power) * rewards[n_to_i[chain[i]]]
        power += 1
    return retrn

In [11]:
chains = [
    ['C1', "C2", "C3", "Pass", "Sleep"],
    ['C1', "FB", "FB", "C1", "C2", "Sleep"],
    ['C1', "C2", "C3", "Pub", "C2", "C3", "Pass", "Sleep"],
    ['C1', "FB", "FB", 'C1', "C2", "C3", "Pub", "C1", "FB", \
     "FB", "FB", "C1", "C2", "C3", "Pub", "C2", "Sleep"],
]
compute_return(0, chains[3], gamma=.5)

-3.196044921875

In [12]:
def compute_value(Pss, rewards, gamma=.05):
    rewards = np.array(rewards).reshape((-1, 1))
    values = np.dot(np.linalg.inv(np.eye(7,7) - gamma * Pss), rewards)
    return values

In [13]:
values = compute_value(Pss, rewards, 0.99999)
values

array([[-12.54073351],
       [  1.45690179],
       [  4.32117045],
       [ 10.        ],
       [  0.80308417],
       [-22.53857963],
       [  0.        ]])

In [14]:
np.eye(7,7)

array([[1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 1.]])

In [15]:
np.linalg.inv(np.eye(7,7))

array([[1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 1.]])